### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

In [ ]:
import zipfile
import os

working_folder = './ Drive/TransformersCode/02-ECommerce/chatbot/'

zip_file_path = working_folder + 'estore_qa.zip'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(working_folder)

In [ ]:
pip install datasets

In [ ]:
file_name_QA = working_folder +  "estore_qa.csv"

from datasets import load_dataset

dataset = load_dataset('csv', data_files={'data': file_name_QA}, split="data")

print(dataset[:1])

In [ ]:
from transformers import AutoTokenizer

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples['question'], examples['answer'], truncation=True, padding='max_length', max_length=128)


tokenized_dataset = dataset.map(tokenize_function, remove_columns=["question", "answer"])

print(tokenized_dataset[:1])

In [ ]:
from transformers import AutoModelForCausalLM, DataCollatorForLanguageModeling, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained(model_name)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
   output_dir=working_folder,
    overwrite_output_dir=True,
    num_train_epochs=20,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset,
)

trainer.train()

model_path = working_folder + "model_dir"

trainer.save_model(model_path)

tokenizer.save_pretrained(model_path)